# 1. IMPORT LIBRARY

In [ ]:
# !pip uninstall -y pytorch-lightning lightning-fabric lightning-utilities torchmetrics darts --quiet

# !pip install -r /kaggle/input/d/januaryfined/enviroment/requirements.txt --quiet

In [ ]:
# !pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu130

## 1.1. Import Library

In [ ]:

import numpy             as np
import pandas            as pd
import matplotlib.pyplot as plt
import seaborn           as sns
import os

from copy import deepcopy

from sklearn.preprocessing import OneHotEncoder  # Encode feature
from sklearn.preprocessing import OrdinalEncoder # Encode feature
from sklearn.preprocessing import MinMaxScaler   # Scale feature
from sklearn.preprocessing import StandardScaler # Scale feature
from sklearn.preprocessing import LabelEncoder   # Encode target
# from scipy.stats import boxcox # Normalized feature

# from sklearn.feature_selection import mutual_info_classif # PCA
from sklearn.model_selection   import train_test_split
from sklearn.model_selection   import RepeatedKFold
from sklearn.model_selection   import GridSearchCV
from sklearn.model_selection   import validation_curve
from sklearn.model_selection   import learning_curve
from sklearn.model_selection   import TimeSeriesSplit

import torch
from darts.models.forecasting.nbeats import NBEATSModel
from darts.models.forecasting.tft_model import TFTModel
import pytorch_lightning
from pytorch_lightning.callbacks import Callback
import json

import darts
from darts import TimeSeries
from darts.metrics import rmse
from darts.utils.data import (
    PastCovariatesSequentialDataset,
    MixedCovariatesSequentialDataset,
    FutureCovariatesSequentialDataset,
    MixedCovariatesInferenceDataset
)

from torch.nn import MSELoss
from pytorch_lightning import Trainer
# %pip install tensorflow
import tensorflow as tf
from tensorflow                 import keras
from tensorflow.keras           import layers
from tensorflow.keras.callbacks import EarlyStopping
from tqdm.auto import tqdm

import joblib
import pickle
import random
import shap


# Classification
# from sklearn.metrics import accuracy_score
# from sklearn.metrics import matthews_corrcoef
# from sklearn.metrics import confusion_matrix
# from sklearn.metrics import roc_auc_score
# from sklearn.metrics import roc_curve
# from sklearn.metrics import classification_report

# Regression
from sklearn.metrics import r2_score
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_squared_log_error
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import mean_absolute_percentage_error
# from sklearn.metrics import median_absolute_error
# from sklearn.metrics import max_error
# from sklearn.metrics import PredictionErrorDisplay

from sklearn.inspection import permutation_importance
from sklearn.tree import plot_tree

In [ ]:
!jupyter nbextension enable --py widgetsnbextension

## 1.2. Choose Models

In [ ]:
# Cấu hình cho nhiều mô hình
# Để thêm mô hình mới, chỉ cần thêm vào dict này
model_configs = {
    'XGBOOST': {
        'type': 'ML',
        'name': 'XGBOOST',
        'alias': 'XGB'
    },
    'RandomForest': {
        'type': 'ML',
        'name': 'Random Forest',
        'alias': 'RF'
    },
    'NBEATS': {
        'type': 'DL',
        'name': 'NBEATS',
        'alias': 'NBEATS'
    },
    # 'Transformer': {
    #     'type': 'DL',
    #     'name': 'Transformer',
    #     'alias': 'TF'
    # },
    'TFT': {
        'type': 'DL',
        'name': 'TFT',
        'alias': 'TFT'
    },
}

current_model = 'NBEATS'  # Chọn mô hình hiện tại ở đây

# Lấy thông tin mô hình hiện tại
model_type    = model_configs[current_model]['type']
model_name    = model_configs[current_model]['name']
model_aliases = model_configs[current_model]['alias']
model_dir    = f"../models/trained_models/{model_type}/{model_name}/"

print(f"📌 Đang làm việc với mô hình: {current_model}")
print(f"   - Type: {model_type}")
print(f"   - Name: {model_name}")
print(f"   - Alias: {model_aliases}")
print(f"   - Path: {model_dir}")

## 1.3. Feature Column

In [ ]:
feature_col = list(['YEAR', 'MONTH', 'DAY',
                    'Nina_index', 'DEW_ave', 'TEMP_ave', 'RH_ave', 
                    # 'AT_ave', 'AT_max',
                    'DEW_max', 'RH_max',
                    'sp_ave', 'tcc_ave', 'tp_sum','ws_ave', 'wd_ave',
                    # 'TEMP_max_1', 'TEMP_max_2', 'TEMP_max_3', 'TEMP_max_4', 'TEMP_max_5', 'TEMP_max_6', 'TEMP_max_7'
                    ])

# 2. DATASET

## 2.1. Loading the dataset

In [ ]:
station = dict({"CaMau" : "CA MAU",
                 "DH"    : "DONG HOI",
                 "NB"    : "NOI BAI",
                 "QN"    : "QUY NHON",
                 "TH"    : "THANH HOA",
                 "TSN"   : "TSN"})

src_name = dict({"CaMau" : "Cà Mau",
                 "DH"    : "Đồng Hới",
                 "NB"    : "Nội Bài",
                 "QN"    : "Quy Nhơn",
                 "TH"    : "Thanh Hóa",
                 "TSN"   : "Tân Sơn Nhất"})

In [ ]:
src_encoded = dict({"CaMau" : "../data/processed/datasets_encoded/CaMau_90.24_encoded.csv",
                    "DH"    : "../data/processed/datasets_encoded/DH_90.24_encoded.csv",
                    "NB"    : "../data/processed/datasets_encoded/NB_90.24_encoded.csv",
                    "QN"    : "../data/processed/datasets_encoded/QN_90.24_encoded.csv",
                    "TH"    : "../data/processed/datasets_encoded/TH_90.24_encoded.csv",
                    "TSN"   : "../data/processed/datasets_encoded/TSN_90.24_encoded.csv"})
# src_encoded = dict({"all" : "../../data/processed/datasets_encoded/all_area_90.24_encoded.csv"})

In [ ]:
# Load dữ liệu encoded cho TỪNG TRẠM vào dictionary
features_encoded = dict()
targets_encoded  = dict() 

for name, link in src_encoded.items():
    df = pd.read_csv(filepath_or_buffer = link,
                     parse_dates        = True,
                     index_col          = "time")
    targets_encoded[name]  = df[['NAME', "TEMP_max"]]
    features_encoded[name] = df.drop(columns=["TEMP_max"])

## 2.2. Spliting the dataset

In [ ]:
# Định nghĩa mốc thời gian
start_train = pd.Timestamp("1990-01-01 00:00:00")
end_train   = pd.Timestamp("2022-12-31 00:00:00")
start_test  = pd.Timestamp("2023-01-01 00:00:00")
end_test    = pd.Timestamp("2024-12-31 00:00:00")

# Khởi tạo dictionaries
X_train_encoded = dict()
X_valid_encoded = dict()
X_test_encoded  = dict()

y_train_encoded = dict()
y_valid_encoded = dict()
y_test_encoded  = dict()

for name in station.keys():
    # Ép index về DatetimeIndex
    features_encoded[name].index = pd.to_datetime(features_encoded[name].index)
    targets_encoded[name].index  = pd.to_datetime(targets_encoded[name].index)
    
    # Tạo mask
    train_mask = (features_encoded[name].index >= start_train) & (features_encoded[name].index <= pd.Timestamp("2020-12-31 00:00:00"))
    valid_mask = (features_encoded[name].index >= pd.Timestamp("2021-01-01 00:00:00")) & (features_encoded[name].index <= end_train)
    test_mask  = (features_encoded[name].index >= start_test) & (features_encoded[name].index <= end_test)
    
    # Chia dữ liệu
    X_train_encoded[name] = features_encoded[name][train_mask]
    X_valid_encoded[name] = features_encoded[name][valid_mask]
    X_test_encoded[name]  = features_encoded[name][test_mask]
    
    y_train_encoded[name] = targets_encoded[name][train_mask]
    y_valid_encoded[name] = targets_encoded[name][valid_mask]
    y_test_encoded[name]  = targets_encoded[name][test_mask]

# 3. MODELS WRAPPER

In [ ]:
class NBEATSWrapper:
    """
    Wrapper sklearn-like cho NBEATSModel (Darts 0.35.0)
    Tách biệt lưu trữ Train và Valid set.
    """

    def __init__(self, model=None, param_dict=None):
        if model is not None:
            self.model = model
            self.param_dict = getattr(model, '_model_params', {}).copy()
        elif param_dict is not None:
            self.model = NBEATSModel(**param_dict)
            self.param_dict = param_dict
        else:
            raise ValueError("Phải truyền model hoặc param_dict")

        self.feature_cols = None
        self.freq = None
        
        # 🚨 LƯU TRỮ TÁCH BIỆT TRAIN VÀ VALID
        self.last_y_train_ts = None
        self.last_X_train_ts = None
        self.last_y_valid_ts = None
        self.last_X_valid_ts = None 
        
        self.feature_importances_ = None
        self.history = {}
   
    # ------------------- sklearn API -------------------
    def get_params(self, deep=True):
        return self.param_dict

    def set_params(self, **params):
        self.param_dict.update(params)
        self.model = NBEATSModel(**self.param_dict)
        self.feature_importances_ = None
        self.last_y_train_ts      = None
        self.last_X_train_ts      = None
        self.last_y_valid_ts      = None
        self.last_X_valid_ts      = None
        self.history              = {}
        return self
    
    
    class LossLogger(Callback):
        def __init__(self):
            self.train_loss = []
            self.val_loss   = []

        # Bắt sự kiện cuối mỗi epoch huấn luyện
        def on_train_epoch_end(self, trainer, pl_module):
            # Lấy giá trị loss từ metrics (Darts thường log key là "train_loss")
            loss = trainer.callback_metrics.get("train_loss")
            if loss is not None:
                self.train_loss.append(loss.item())

        # Bắt sự kiện cuối mỗi epoch validation
        def on_validation_epoch_end(self, trainer, pl_module):
            loss = trainer.callback_metrics.get("val_loss")
            if loss is not None:
                self.val_loss.append(loss.item())
                
    # ------------------- FIT -------------------
    def fit(self, X, y, X_valid=None, y_valid=None, verbose=True, fi=False, loss=False):
    
        # 1. Store feature columns và Chuẩn hóa X/y về DataFrame/Series
        if isinstance(X, np.ndarray):
            X = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(X.shape[1])])
            
        self.feature_cols = X.columns.tolist()
        
        # 2. Convert to TimeSeries (Bắt buộc để lấy freq)
        y_ts = TimeSeries.from_series(y)
        self.freq = y_ts.freq_str 
        
        if self.freq is None:
            raise ValueError("Lỗi: Không thể xác định tần suất (frequency) của TimeSeries. Đảm bảo Index là DatetimeIndex liên tục.")
    
        # Đảm bảo X có DatetimeIndex
        if not isinstance(X.index, pd.DatetimeIndex):
             X.index = pd.date_range(start=y.index[0], periods=len(X), freq=self.freq)
        
        X_ts = TimeSeries.from_dataframe(X, freq=self.freq)
        
        # Xử lý Validation Set
        y_valid_ts, X_valid_ts = None, None
        if X_valid is not None and y_valid is not None:
            if isinstance(X_valid, np.ndarray):
                 X_valid = pd.DataFrame(X_valid, columns=self.feature_cols)
    
            y_valid_ts = TimeSeries.from_series(y_valid, freq=self.freq)
            
            if not isinstance(X_valid.index, pd.DatetimeIndex):
                 X_valid.index = pd.date_range(start=y_valid.index[0], periods=len(X_valid), freq=self.freq)
                 
            X_valid_ts = TimeSeries.from_dataframe(X_valid, freq=self.freq)
        
        # =========================================================
        # 🚨 CALLBACK ĐỂ LẤY LOSS
        # =========================================================
        loss_logger = self.LossLogger()

        device = torch.cuda.is_available()
        trainer = None
        if device:
            trainer = Trainer(accelerator         = 'gpu',
                              devices             = torch.cuda.device_count(), # Mặc định GPU
                              max_epochs          = self.get_params()["n_epochs"],
                              precision           = "64-true",
                              enable_progress_bar = True,
                              logger              = True,
                              callbacks           = [loss_logger] if loss==True else [])
            print(f"Sử dụng chế độ {torch.cuda.device_count()} GPU cho dự báo.")
        else:
            trainer = Trainer(accelerator         = 'cpu',
                              devices             = 1, # Mặc định CPU
                              precision           = "64-true",
                              enable_progress_bar = True,
                              logger              = True,
                              callbacks           = [loss_logger] if loss==True else [])
            print(f"Sử dụng chế độ CPU cho dự báo.")
        
        
        # 3. Fit Model
        self.model.fit(
            series              = y_ts,
            past_covariates     = X_ts,
            # past_covariates     = None,
            val_series          = y_valid_ts,
            val_past_covariates = X_valid_ts,
            trainer             = trainer,
            verbose             = verbose
        )
    
        self.history["train_loss"] = loss_logger.train_loss
        self.history["val_loss"]   = loss_logger.val_loss
        
        if verbose:
            print(f"Train Loss (Final): {loss_logger.train_loss[-1] if loss_logger.train_loss else 'N/A'}")
            print(f"Val Loss (Final)  : {loss_logger.val_loss[-1] if loss_logger.val_loss else 'N/A'}")
    
        # 4. 🚨 Lưu lịch sử TÁCH BIỆT Train và Valid
        self.last_y_train_ts = y_ts
        self.last_X_train_ts = X_ts
        self.last_y_valid_ts = y_valid_ts
        self.last_X_valid_ts = X_valid_ts
    
        if fi:
            try:
                if verbose:
                    print(">> Đang tính Feature Importance (Permutation Method)...")

                # A. Xác định tập dữ liệu để đánh giá (ưu tiên Valid Set)
                if X_valid_ts is not None and y_valid_ts is not None:
                    # 🚨 Logic nối chuỗi: Lấy đoạn cuối Train + Valid để model có đủ context
                    start_lookback = y_ts.end_time() - pd.Timedelta(self.model.input_chunk_length - 1, unit=self.freq)

                    eval_series = y_ts.slice(start_lookback, y_ts.end_time()).concatenate(y_valid_ts, axis=0)
                    eval_covariates = X_ts.slice(start_lookback, X_ts.end_time()).concatenate(X_valid_ts, axis=0)
                else:
                    # Nếu không có Valid, dùng Train
                    eval_series = y_ts
                    eval_covariates = X_ts

                # B. Tính Baseline RMSE (Dự báo chuẩn không xáo trộn)
                baseline_preds = self.model.historical_forecasts(
                    series           = eval_series,
                    past_covariates  = eval_covariates,
                    start            = self.model.input_chunk_length,
                    forecast_horizon = 1,
                    retrain          = False,
                    verbose          = False,
                    # trainer         = trainer
                )

                # Cắt target thực tế khớp với đoạn dự báo
                intersect_target = eval_series.slice(baseline_preds.start_time(), baseline_preds.end_time())
                baseline_score = rmse(intersect_target, baseline_preds)

                # C. Loop Permutation (Xáo trộn từng feature)
                importances = []
                df_cov_orig = pd.DataFrame(eval_covariates.values(),
                                       index=eval_covariates.time_index,
                                       columns=self.feature_cols)

                for col in self.feature_cols:
                    # Copy và shuffle cột col
                    df_shuffled = df_cov_orig.copy()
                    df_shuffled[col] = np.random.permutation(df_shuffled[col].values)
                    ts_shuffled = TimeSeries.from_dataframe(df_shuffled, freq=self.freq)

                    # Dự báo lại
                    shuffled_preds = self.model.historical_forecasts(
                        series           = eval_series,
                        past_covariates  = ts_shuffled,
                        start            = self.model.input_chunk_length,
                        forecast_horizon = 1,
                        retrain          = False,
                        verbose          = False,
                        # trainer        = trainer
                    )

                    score = rmse(intersect_target, shuffled_preds)
                    # Importance = Lỗi tăng thêm
                    importances.append(max(0, score - baseline_score))

                # D. Normalize về 1
                imp_arr = np.array(importances)
                if imp_arr.sum() > 0:
                    self.feature_importances_ = imp_arr / imp_arr.sum()
                else:
                    self.feature_importances_ = np.zeros(len(self.feature_cols))

                if verbose:
                    print(">> Tính Feature Importance hoàn tất.")

            except Exception as e:
                if verbose:
                    print(f"Cảnh báo: Lỗi khi tính Feature Importance. Đảm bảo NBEATSTExplainer đã được cài đặt đúng cách và dữ liệu đủ dài. Chi tiết: {e}")
                self.feature_importances_ = None
    
        return self

    # ------------------- PREDICT -------------------
    def predict(self, 
                X, 
                use_valid_history = True, 
                verbose           = True, 
                n_jobs            = -1, 
                rolling: bool     =  True):
        
        import warnings, sys, copy, torch, logging
        warnings.filterwarnings("ignore")
    
        # ===== 0. TẮT TOÀN BỘ LOG LIGHTNING =====
        # Logic này giúp tắt các thông báo của PyTorch Lightning
        try:
            import pytorch_lightning as pl
            pl.utilities.rank_zero._get_rank      = lambda: 1
            pl.utilities.rank_zero.rank_zero_only = lambda *a, **k: (lambda f: f)
            pl.utilities.rank_zero.rank_zero_info = lambda *a, **k: None
            pl.utilities.rank_zero.rank_zero_warn = lambda *a, **k: None
    
            logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
            logging.getLogger("lightning_fabric").setLevel(logging.ERROR)
            logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)
    
            pl.loggers.base.LightningLoggerBase.info  = lambda *a, **k: None
            pl.loggers.base.LightningLoggerBase.warn  = lambda *a, **k: None
            pl.loggers.base.LightningLoggerBase.debug = lambda *a, **k: None
    
        except Exception:
            pass
        
        # ===== 1. BUILD HISTORY =====
        if self.freq is None:
            raise RuntimeError("Cần gọi fit() trước để thiết lập freq.")
    
        if self.last_y_train_ts is None or self.last_X_train_ts is None:
            raise RuntimeError("Thiếu history để dự báo.")
    
        y_hist = self.last_y_train_ts
        X_hist = self.last_X_train_ts
    
        if use_valid_history and self.last_y_valid_ts is not None:
            y_hist = y_hist.concatenate(self.last_y_valid_ts, axis=0)
            X_hist = X_hist.concatenate(self.last_X_valid_ts, axis=0)
    
        # ===== 2. CONVERT X → TimeSeries =====
        # Xử lý input (np.ndarray, DataFrame, TimeSeries) và chuyển thành TimeSeries (X_test_ts)
        if isinstance(X, np.ndarray):
            if self.feature_cols is None:
                raise RuntimeError("self.feature_cols chưa được thiết lập. Run fit() first.")
            X_df = pd.DataFrame(X, columns=self.feature_cols)
            # Gán index thời gian bắt đầu từ điểm cuối của lịch sử + 1
            start_date = y_hist.end_time() + pd.Timedelta(1, unit=self.freq)
            X_df.index = pd.date_range(start   = start_date, 
                                       periods = len(X_df), 
                                       freq    = self.freq)
            X_test_ts = TimeSeries.from_dataframe(X_df, freq=self.freq)
    
        elif isinstance(X, pd.DataFrame):
            if not isinstance(X.index, pd.DatetimeIndex):
                start_date = y_hist.end_time() + pd.Timedelta(1, unit=self.freq)
                X.index = pd.date_range(start   = start_date, 
                                        periods = len(X), 
                                        freq    = self.freq)
            X_test_ts = TimeSeries.from_dataframe(X, freq=self.freq)
    
        elif isinstance(X, TimeSeries):
            X_test_ts = X
    
        else:
            raise TypeError("X phải là numpy array, DataFrame hoặc TimeSeries.")
    
        # ===== 3. SETUP TRAINER & ACCELERATOR LOGIC =====
        n = len(X_test_ts)
        preds = []
        input_len = self.get_params()["input_chunk_length"]
    
        from pytorch_lightning import Trainer
    
    
        device = torch.cuda.is_available()
        use_progress_bar_update = True # Mặc định dùng thanh tiến trình cuốn chiếu (cho CPU)
        trainer = None
        if device:
            trainer = Trainer(accelerator         = 'gpu',
                              devices             = torch.cuda.device_count(), # Mặc định GPU
                              precision           = "64-true",
                              enable_progress_bar = False,
                              logger              = False)
            print(f"Sử dụng chế độ {torch.cuda.device_count()} GPU cho dự báo.")
            use_progress_bar_update = False # Chuyển sang in trạng thái 1/10
        else:
            trainer = Trainer(accelerator         = 'cpu',
                              devices             = 1, # Mặc định CPU
                              precision           = "64-true",
                              enable_progress_bar = False,
                              logger              = False)
            print("Cảnh báo: Không tìm thấy GPU, đang dùng chế độ CPU cho dự báo.")
            print(f"Sử dụng chế độ CPU cho dự báo.")
    
    
        # ===== 4. PREDICTION LOGIC =====
        if not rolling:
            # --- Dự đoán cả batch (Fast Prediction) ---
            # Chỉ lấy phần history covariates (cho encoder) cần thiết
            X_hist_part = X_hist.tail(input_len)

            # Ghép lịch sử X (cho encoder) + toàn bộ X_test_ts (cho decoder)
            X_full_ts = X_hist_part.concatenate(X_test_ts, axis=0)
            
            # Dự đoán toàn bộ n bước một lần
            y_pred_ts = self.model.predict(
                n               = n,
                series          = y_hist,
                past_covariates = X_full_ts,
                verbose         = False,
                n_jobs          = n_jobs,
                trainer         = trainer
            )
            
            # Lấy giá trị của target column đầu tiên
            preds = y_pred_ts.values()[:, 0] 
            
            if verbose:
                print(f"Dự báo batch {n} mẫu hoàn tất!")
            return np.array(preds)

        else:
            # --- Rolling forecast từng bước ---
            progress_step = max(1, n // 10) # Tính bước cho cập nhật 1/10

            for i in range(n):
                if verbose:
                    if use_progress_bar_update:
                        # CPU style: Incremental update
                        sys.stdout.write(f"\rĐang dự báo: {i+1}/{n}")
                        sys.stdout.flush()
                    elif (i + 1) % progress_step == 0 or i == n - 1:
                        # GPU style: Print status every 1/10 steps
                        percent = (i + 1) / n * 100
                        print(f"[{'CUDA' if torch.cuda.is_available() else 'CPU'}] Đang dự báo cuốn chiếu: {i+1}/{n} ({percent:.0f}%)")
        
                # Lấy 1 timestep của past covariates   cần dự báo
                Xi = X_test_ts[i:i+1]
                
                # Lấy phần history covariates (cho encoder) cần thiết
                X_hist_part = X_hist.tail(input_len)
                
                # Past covariates   cho Darts predict phải bao gồm cả phần history cho encoder và điểm dự báo
                X_full_ts = X_hist_part.concatenate(Xi, axis=0)
        
                # Thực hiện dự đoán 1 bước (n=1)
                y_pred_step = self.model.predict(
                    n               = 1,
                    series          = y_hist, # Lịch sử output (y)
                    past_covariates = X_full_ts, # Lịch sử X + điểm X hiện tại
                    n_jobs          = n_jobs, 
                    verbose         = False,
                    trainer         = trainer
                )
                preds.append(y_pred_step.values()[0][0])
        
                # Cập nhật lịch sử (cuốn chiếu)
                y_hist = y_hist.concatenate(y_pred_step, axis=0)
                X_hist = X_hist.concatenate(Xi, axis=0)
        
            if verbose:
                # Xóa dòng tiến trình cuối cùng (chỉ áp dụng cho CPU style)
                if use_progress_bar_update:
                    sys.stdout.write("\rDự báo hoàn tất!                                 \n")
                    sys.stdout.flush()
                else:
                    print(f"[{'CUDA' if torch.cuda.is_available() else 'CPU'}] Dự báo hoàn tất!")
        
            return np.array(preds)



    def predict_history(self, type="train", verbose=True):
        """
        Tái tạo dự báo trên dữ liệu lịch sử (Train hoặc Valid).
        type: "train" hoặc "valid"
        """
        if type == "train":
            y_hist_ts = self.last_y_train_ts
            X_hist_ts = self.last_X_train_ts
        elif type == "valid":
            y_hist_ts = self.last_y_valid_ts
            X_hist_ts = self.last_X_valid_ts
        else:
            raise ValueError("type phải là 'train' hoặc 'valid'.")

        if y_hist_ts is None or X_hist_ts is None:
            print(f"Cảnh báo: Không tìm thấy dữ liệu lịch sử cho type='{type}'. Bỏ qua predict_history.")
            return None
            
        # 2. Chạy historical_forecasts
        forecasts_ts_list = self.model.historical_forecasts(
            series            = y_hist_ts,
            past_covariates   = X_hist_ts, 
            start             = self.model.input_chunk_length,
            forecast_horizon  = self.model.output_chunk_length, 
            retrain           = False,
            last_points_only  = True,
            verbose           = verbose
        )
        
        # historical_forecasts trả về TimeSeries hoặc List[TimeSeries]
        if isinstance(forecasts_ts_list, TimeSeries):
            return forecasts_ts_list.values().flatten()
        else:
            # Darts 0.28.0 cho phép .values().flatten() trực tiếp trên List[TimeSeries]
            return TimeSeries.concatenate(forecasts_ts_list, axis=0).values().flatten()



class TFTWrapper:
    """
    Wrapper sklearn-like cho TFTModel (Darts 0.35.0)
    Tách biệt lưu trữ Train và Valid set.
    """

    def __init__(self, model=None, param_dict=None):
        if model is not None:
            self.model = model
            self.param_dict = getattr(model, '_model_params', {}).copy()
        elif param_dict is not None:
            self.model = TFTModel(**param_dict)
            self.param_dict = param_dict
        else:
            raise ValueError("Phải truyền model hoặc param_dict")

        self.feature_cols = None
        self.freq = None
        
        # 🚨 LƯU TRỮ TÁCH BIỆT TRAIN VÀ VALID
        self.last_y_train_ts = None
        self.last_X_train_ts = None
        self.last_y_valid_ts = None
        self.last_X_valid_ts = None 
        
        self.feature_importances_ = None
        self.history = {}
   
    # ------------------- sklearn API -------------------
    def get_params(self, deep=True):
        return self.param_dict

    def set_params(self, **params):
        self.param_dict.update(params)
        self.model = TFTModel(**self.param_dict)
        self.last_y_train_ts      = None
        self.last_X_train_ts      = None
        self.last_y_valid_ts      = None
        self.last_X_valid_ts      = None
        self.feature_importances_ = None
        self.history              = {}
        return self


    class LossLogger(Callback):
        def __init__(self):
            self.train_loss = []
            self.val_loss   = []

        # Bắt sự kiện cuối mỗi epoch huấn luyện
        def on_train_epoch_end(self, trainer, pl_module):
            # Lấy giá trị loss từ metrics (Darts thường log key là "train_loss")
            loss = trainer.callback_metrics.get("train_loss")
            if loss is not None:
                self.train_loss.append(loss.item())

        # Bắt sự kiện cuối mỗi epoch validation
        def on_validation_epoch_end(self, trainer, pl_module):
            loss = trainer.callback_metrics.get("val_loss")
            if loss is not None:
                self.val_loss.append(loss.item())

    # ------------------- FIT -------------------
    def fit(self, X, y, X_valid=None, y_valid=None, verbose=True, fi=False, loss=False):
    
        # 1. Store feature columns và Chuẩn hóa X/y về DataFrame/Series
        if isinstance(X, np.ndarray):
            X = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(X.shape[1])])
            
        self.feature_cols = X.columns.tolist()
        
        # 2. Convert to TimeSeries (Bắt buộc để lấy freq)
        y_ts = TimeSeries.from_series(y)
        self.freq = y_ts.freq_str 
        
        if self.freq is None:
            raise ValueError("Lỗi: Không thể xác định tần suất (frequency) của TimeSeries. Đảm bảo Index là DatetimeIndex liên tục.")
    
        # Đảm bảo X có DatetimeIndex
        if not isinstance(X.index, pd.DatetimeIndex):
             X.index = pd.date_range(start=y.index[0], periods=len(X), freq=self.freq)
        
        X_ts = TimeSeries.from_dataframe(X, freq=self.freq)
        
        # Xử lý Validation Set
        y_valid_ts, X_valid_ts = None, None
        if X_valid is not None and y_valid is not None:
            if isinstance(X_valid, np.ndarray):
                 X_valid = pd.DataFrame(X_valid, columns=self.feature_cols)
    
            y_valid_ts = TimeSeries.from_series(y_valid, freq=self.freq)
            
            if not isinstance(X_valid.index, pd.DatetimeIndex):
                 X_valid.index = pd.date_range(start=y_valid.index[0], periods=len(X_valid), freq=self.freq)
                 
            X_valid_ts = TimeSeries.from_dataframe(X_valid, freq=self.freq)

        # =========================================================
        # 🚨 CALLBACK ĐỂ LẤY LOSS
        # =========================================================
        loss_logger = self.LossLogger()
        
        device = torch.cuda.is_available()
        trainer = None
        if device:
            trainer = Trainer(accelerator         = 'gpu',
                              devices             = torch.cuda.device_count(), # Mặc định GPU
                              max_epochs          = self.get_params()["n_epochs"],
                              precision           = "64-true",
                              enable_progress_bar = True,
                              logger              = True,
                              callbacks           = [loss_logger] if loss==True else [])
            print(f"Sử dụng chế độ {torch.cuda.device_count()} GPU cho dự báo.")
        else:
            trainer = Trainer(accelerator         = 'cpu',
                              devices             = 1, # Mặc định CPU
                              precision           = "64-true",
                              enable_progress_bar = True,
                              logger              = True,
                              callbacks           = [loss_logger] if loss==True else [])
            print(f"Sử dụng chế độ CPU cho dự báo.")
        
        
        # 3. Fit Model
        self.model.fit(
            series                = y_ts,
            past_covariates       = None,
            future_covariates     = X_ts,
            val_series            = y_valid_ts,
            val_future_covariates = X_valid_ts,
            trainer               = trainer,
            verbose               = verbose
        )
    
        self.history["train_loss"] = loss_logger.train_loss
        self.history["val_loss"]   = loss_logger.val_loss
        
        if verbose:
            print(f"Train Loss (Final): {loss_logger.train_loss[-1] if loss_logger.train_loss else 'N/A'}")
            print(f"Val Loss (Final)  : {loss_logger.val_loss[-1] if loss_logger.val_loss else 'N/A'}")
        
        # 4. 🚨 Lưu lịch sử TÁCH BIỆT Train và Valid
        self.last_y_train_ts = y_ts
        self.last_X_train_ts = X_ts
        self.last_y_valid_ts = y_valid_ts
        self.last_X_valid_ts = X_valid_ts
    
        # 5. Compute feature importance (TFTExplainer)
        if fi:
            try:
                # 1. TẠO CHUỖI FUTURE COVARIATES ĐÃ MỞ RỘNG (X_ts_plus_one)
                X_ts_plus_one = None
                if X_valid_ts is not None:
                    # Nếu có Valid, nối Train X và điểm đầu tiên của Valid X
                    X_T_plus_1 = X_valid_ts.head(1) 
                    X_ts_plus_one = X_ts.concatenate(X_T_plus_1, axis=0)
                else:
                    # 🚨 FIX Logic: Tạo một chuỗi NaN có Index đúng và nối vào X_ts
                    nan_ts = TimeSeries.from_times_and_values(
                        times=pd.date_range(start   = X_ts.end_time() + pd.Timedelta(1, unit=self.freq), 
                                            periods = 1, 
                                            freq    = self.freq),
                        values=np.full((1, len(self.feature_cols)), np.nan)
                    )
                    X_ts_plus_one = X_ts.concatenate(nan_ts, axis=0)
                    
                # 2. KHỞI TẠO EXPLAINER VÀ TÍNH TOÁN
                explainer = TFTExplainer(
                    model                        = self.model,
                    background_series            = y_ts,
                    background_future_covariates = X_ts_plus_one
                )
                
                res = explainer.explain().get_feature_importances()
                encoder_imp = res.get('encoder_importance', pd.DataFrame())
                
                # 3. TRÍCH XUẤT VÀ CHUẨN HÓA KẾT QUẢ
                imp_list = []
                for col in self.feature_cols:
                    matched_cols = [c for c in encoder_imp.columns if c.startswith(col)]
                    if matched_cols:
                        # Lấy giá trị đầu tiên (giả định importance là 1 chiều)
                        imp_list.append(encoder_imp[matched_cols[0]].values[0])
                    else:
                        imp_list.append(0.0)
                
                imp_array = np.array(imp_list, dtype=float)
                if imp_array.sum() > 0:
                    imp_array = imp_array / imp_array.sum()
                self.feature_importances_ = imp_array
                
            except Exception as e:
                if verbose:
                    print(f"Cảnh báo: Lỗi khi tính Feature Importance. Đảm bảo TFTExplainer đã được cài đặt đúng cách và dữ liệu đủ dài. Chi tiết: {e}")
                self.feature_importances_ = None
    
        return self

    # ------------------- PREDICT -------------------
    def predict(self, 
                X, 
                use_valid_history = True, 
                verbose           = True, 
                n_jobs            = -1, 
                rolling: bool     =  True):
        
        import warnings, sys, copy, torch, logging
        warnings.filterwarnings("ignore")
    
        # ===== 0. TẮT TOÀN BỘ LOG LIGHTNING =====
        # Logic này giúp tắt các thông báo của PyTorch Lightning
        try:
            import pytorch_lightning as pl
            pl.utilities.rank_zero._get_rank      = lambda: 1
            pl.utilities.rank_zero.rank_zero_only = lambda *a, **k: (lambda f: f)
            pl.utilities.rank_zero.rank_zero_info = lambda *a, **k: None
            pl.utilities.rank_zero.rank_zero_warn = lambda *a, **k: None
    
            logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
            logging.getLogger("lightning_fabric").setLevel(logging.ERROR)
            logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)
    
            pl.loggers.base.LightningLoggerBase.info  = lambda *a, **k: None
            pl.loggers.base.LightningLoggerBase.warn  = lambda *a, **k: None
            pl.loggers.base.LightningLoggerBase.debug = lambda *a, **k: None
    
        except Exception:
            pass
        
        # ===== 1. BUILD HISTORY =====
        if self.freq is None:
            raise RuntimeError("Cần gọi fit() trước để thiết lập freq.")
    
        if self.last_y_train_ts is None or self.last_X_train_ts is None:
            raise RuntimeError("Thiếu history để dự báo.")
    
        y_hist = self.last_y_train_ts
        X_hist = self.last_X_train_ts
    
        if use_valid_history and self.last_y_valid_ts is not None:
            y_hist = y_hist.concatenate(self.last_y_valid_ts, axis=0)
            X_hist = X_hist.concatenate(self.last_X_valid_ts, axis=0)
    
        # ===== 2. CONVERT X → TimeSeries =====
        # Xử lý input (np.ndarray, DataFrame, TimeSeries) và chuyển thành TimeSeries (X_test_ts)
        if isinstance(X, np.ndarray):
            if self.feature_cols is None:
                raise RuntimeError("self.feature_cols chưa được thiết lập. Run fit() first.")
            X_df = pd.DataFrame(X, columns=self.feature_cols)
            # Gán index thời gian bắt đầu từ điểm cuối của lịch sử + 1
            start_date = y_hist.end_time() + pd.Timedelta(1, unit=self.freq)
            X_df.index = pd.date_range(start   = start_date, 
                                       periods = len(X_df), 
                                       freq    = self.freq)
            X_test_ts = TimeSeries.from_dataframe(X_df, freq=self.freq)
    
        elif isinstance(X, pd.DataFrame):
            if not isinstance(X.index, pd.DatetimeIndex):
                start_date = y_hist.end_time() + pd.Timedelta(1, unit=self.freq)
                X.index = pd.date_range(start   = start_date, 
                                        periods = len(X), 
                                        freq    = self.freq)
            X_test_ts = TimeSeries.from_dataframe(X, freq=self.freq)
    
        elif isinstance(X, TimeSeries):
            X_test_ts = X
    
        else:
            raise TypeError("X phải là numpy array, DataFrame hoặc TimeSeries.")
    
        # ===== 3. SETUP TRAINER & ACCELERATOR LOGIC =====
        n = len(X_test_ts)
        preds = []
        input_len = self.get_params()["input_chunk_length"]
    
        from pytorch_lightning import Trainer
    
    
        device = torch.cuda.is_available()
        use_progress_bar_update = True # Mặc định dùng thanh tiến trình cuốn chiếu (cho CPU)
        trainer = None
        if device:
            trainer = Trainer(accelerator         = 'gpu',
                              devices             = torch.cuda.device_count(), # Mặc định GPU
                              precision           = "64-true",
                              enable_progress_bar = False,
                              logger              = False)
            print(f"Sử dụng chế độ {torch.cuda.device_count()} GPU cho dự báo.")
            use_progress_bar_update = False # Chuyển sang in trạng thái 1/10
        else:
            trainer = Trainer(accelerator         = 'cpu',
                              devices             = 1, # Mặc định CPU
                              precision           = "64-true",
                              enable_progress_bar = False,
                              logger              = False)
            print("Cảnh báo: Không tìm thấy GPU, đang dùng chế độ CPU cho dự báo.")
            print(f"Sử dụng chế độ CPU cho dự báo.")
    
    
        # ===== 4. PREDICTION LOGIC =====
        if not rolling:
            # --- Dự đoán cả batch (Fast Prediction) ---
            # Chỉ lấy phần history covariates (cho encoder) cần thiết
            X_hist_part = X_hist.tail(input_len)

            # Ghép lịch sử X (cho encoder) + toàn bộ X_test_ts (cho decoder)
            X_full_ts = X_hist_part.concatenate(X_test_ts, axis=0)
            
            # Dự đoán toàn bộ n bước một lần
            y_pred_ts = self.model.predict(
                n                 = n,
                series            = y_hist,
                future_covariates = X_full_ts,
                verbose           = False,
                n_jobs            = n_jobs,
                trainer           = trainer
            )
            
            # Lấy giá trị của target column đầu tiên
            preds = y_pred_ts.values()[:, 0] 
            
            if verbose:
                print(f"Dự báo batch {n} mẫu hoàn tất!")
            return np.array(preds)

        else:
            # --- Rolling forecast từng bước ---
            progress_step = max(1, n // 10) # Tính bước cho cập nhật 1/10

            for i in range(n):
                if verbose:
                    if use_progress_bar_update:
                        # CPU style: Incremental update
                        sys.stdout.write(f"\rĐang dự báo: {i+1}/{n}")
                        sys.stdout.flush()
                    elif (i + 1) % progress_step == 0 or i == n - 1:
                        # GPU style: Print status every 1/10 steps
                        percent = (i + 1) / n * 100
                        print(f"[{'CUDA' if torch.cuda.is_available() else 'CPU'}] Đang dự báo cuốn chiếu: {i+1}/{n} ({percent:.0f}%)")
        
                # Lấy 1 timestep của future covariates cần dự báo
                Xi = X_test_ts[i:i+1]
                
                # Lấy phần history covariates (cho encoder) cần thiết
                X_hist_part = X_hist.tail(input_len)
                
                # Future covariates cho Darts predict phải bao gồm cả phần history cho encoder và điểm dự báo
                X_full_ts = X_hist_part.concatenate(Xi, axis=0)
        
                # Thực hiện dự đoán 1 bước (n=1)
                y_pred_step = self.model.predict(
                    n                 = 1,
                    series            = y_hist, # Lịch sử output (y)
                    future_covariates = X_full_ts, # Lịch sử X + điểm X hiện tại
                    n_jobs            = n_jobs, 
                    verbose           = False,
                    trainer           = trainer
                )
                preds.append(y_pred_step.values()[0][0])
        
                # Cập nhật lịch sử (cuốn chiếu)
                y_hist = y_hist.concatenate(y_pred_step, axis=0)
                X_hist = X_hist.concatenate(Xi, axis=0)
        
            if verbose:
                # Xóa dòng tiến trình cuối cùng (chỉ áp dụng cho CPU style)
                if use_progress_bar_update:
                    sys.stdout.write("\rDự báo hoàn tất!                                 \n")
                    sys.stdout.flush()
                else:
                    print(f"[{'CUDA' if torch.cuda.is_available() else 'CPU'}] Dự báo hoàn tất!")
        
            return np.array(preds)



    def predict_history(self, type="train", verbose=True):
        """
        Tái tạo dự báo trên dữ liệu lịch sử (Train hoặc Valid).
        type: "train" hoặc "valid"
        """
        if type == "train":
            y_hist_ts = self.last_y_train_ts
            X_hist_ts = self.last_X_train_ts
        elif type == "valid":
            y_hist_ts = self.last_y_valid_ts
            X_hist_ts = self.last_X_valid_ts
        else:
            raise ValueError("type phải là 'train' hoặc 'valid'.")

        if y_hist_ts is None or X_hist_ts is None:
            print(f"Cảnh báo: Không tìm thấy dữ liệu lịch sử cho type='{type}'. Bỏ qua predict_history.")
            return None
            
        # 2. Chạy historical_forecasts
        forecasts_ts_list = self.model.historical_forecasts(
            series            = y_hist_ts,
            future_covariates = X_hist_ts, 
            start             = self.model.input_chunk_length,
            forecast_horizon  = self.model.output_chunk_length, 
            retrain           = False,
            last_points_only  = True,
            verbose           = verbose
        )
        
        # historical_forecasts trả về TimeSeries hoặc List[TimeSeries]
        if isinstance(forecasts_ts_list, TimeSeries):
            return forecasts_ts_list.values().flatten()
        else:
            # Darts 0.28.0 cho phép .values().flatten() trực tiếp trên List[TimeSeries]
            return TimeSeries.concatenate(forecasts_ts_list, axis=0).values().flatten() 


class RFWrapper:
    """
    Wrapper sklearn-like cho RandomForestRegressor.
    Fix lỗi: Setting an array element with a sequence.
    """

    def __init__(self, model=None, param_dict=None, input_chunk_length=7, output_chunk_length=1):
        if model is not None:
            self.model = model
        elif param_dict is not None:
            self.param_dict = param_dict
            self.model = RandomForestRegressor(**self.param_dict)
        else:
            raise ValueError("Phải truyền model hoặc param_dict")

        self.input_chunk_length  = input_chunk_length
        self.output_chunk_length = output_chunk_length
        
        # Lưu trữ data
        self.last_y_train = None
        self.last_X_train = None
        self.last_y_valid = None
        self.last_X_valid = None 
        self.feature_cols = []

    # ------------------- sklearn API -------------------
    def get_params(self, deep=True):
        return {"input_chunk_length"  : self.input_chunk_length,
                "output_chunk_length" : self.output_chunk_length,
                **self.model.get_params()}

    def set_params(self, **params):
        if "input_chunk_length" in params:
            self.input_chunk_length = params.pop("input_chunk_length")
        if "output_chunk_length" in params:
            self.output_chunk_length = params.pop("output_chunk_length")
        
        self.param_dict.update(**params)
        self.model = RandomForestRegressor(**self.param_dict)
        
        # Lưu trữ data
        self.last_y_train = None
        self.last_X_train = None
        self.last_y_valid = None
        self.last_X_valid = None 
        self.feature_cols = []
        
        return self

    # ------------------- FIT -------------------
    def fit(self, X, y, X_valid=None, y_valid=None, verbose=True, fi=False):
        # 1. Chuẩn hóa về Pandas để dễ xử lý
        if isinstance(X, np.ndarray):
            X = pd.DataFrame(X, columns=[f'feat_{i}' for i in range(X.shape[1])])
        if isinstance(y, np.ndarray):
            y = pd.Series(y)
            
        self.feature_cols = X.columns.tolist()

        # 2. Lưu trữ dữ liệu gốc
        self.last_y_train = y
        self.last_X_train = X
        self.last_y_valid = y_valid
        self.last_X_valid = X_valid

        # 3. Chuẩn bị dữ liệu Numpy
        # Ép kiểu float để tránh lỗi object (như bài trước)
        X_vals = X.values.astype(np.float64)
        y_vals = y.values.astype(np.float64).flatten()
        
        # --- FIX LỖI INDEX ERROR TẠI ĐÂY ---
        # Kiểm tra độ lệch độ dài
        len_x = len(X_vals)
        len_y = len(y_vals)
        
        if len_x != len_y:
            print(f"⚠️ Cảnh báo: Kích thước X ({len_x}) và y ({len_y}) không khớp!")
            # Lấy độ dài chung nhỏ nhất để an toàn
            min_len = min(len_x, len_y)
            X_vals = X_vals[:min_len]
            y_vals = y_vals[:min_len]
            n_samples = min_len
        else:
            n_samples = len_x
        # -----------------------------------

        n_features = X_vals.shape[1]
        
        # Tính toán index loop
        start_idx = self.input_chunk_length
        end_idx = n_samples - self.output_chunk_length + 1
        num_train_samples = end_idx - start_idx
        
        if num_train_samples <= 0:
            raise ValueError(f"Dữ liệu quá ngắn ({n_samples}) so với input ({self.input_chunk_length}) + output ({self.output_chunk_length}).")

        # Pre-allocate arrays
        input_dim = (self.input_chunk_length * n_features) + self.input_chunk_length
        X_train_rf = np.zeros((num_train_samples, input_dim), dtype=np.float64)
        
        if self.output_chunk_length == 1:
            y_train_rf = np.zeros((num_train_samples,), dtype=np.float64)
        else:
            y_train_rf = np.zeros((num_train_samples, self.output_chunk_length), dtype=np.float64)

        if verbose: 
            print(f">> Tạo Rolling Data: Samples={num_train_samples} | Input={self.input_chunk_length} | Output={self.output_chunk_length}")

        # 4. Tạo dữ liệu Rolling
        idx = 0
        for i in range(start_idx, end_idx):
            # --- Input ---
            # X lags
            x_window = X_vals[i - self.input_chunk_length : i].flatten()
            # y lags
            y_window = y_vals[i - self.input_chunk_length : i]
            
            X_train_rf[idx] = np.concatenate([x_window, y_window])
            
            # --- Target ---
            target = y_vals[i : i + self.output_chunk_length]
            
            # (Safety check thừa nhưng an toàn)
            if len(target) == 0:
                raise IndexError(f"Lỗi logic: Target rỗng tại index {i}. Check len(y)={len(y_vals)}")

            if self.output_chunk_length == 1:
                y_train_rf[idx] = target[0] 
            else:
                y_train_rf[idx] = target
                
            idx += 1

        # 5. Fit
        if verbose: print(f">> Fitting RandomForest...")
        self.model.fit(X_train_rf, y_train_rf)
        
        if fi:
            self._calculate_feature_importance(n_features)
                
        return self

    def _calculate_feature_importance(self, n_features_X):
        """Tách logic FI ra hàm riêng cho gọn"""
        try:
            all_lags_importance = self.model.feature_importances_
            input_len = self.input_chunk_length
            
            # Chỉ tính phần X lags (bỏ qua phần y lags ở đuôi vector input)
            # Vector input cấu trúc: [X_lag_1...X_lag_N, y_lag_1...y_lag_N]
            # Độ dài phần X là: n_features_X * input_len
            n_lags_X = n_features_X * input_len
            
            feature_importance_values = np.zeros(n_features_X, dtype=float)
            
            for i in range(n_lags_X):
                col_index = i % n_features_X
                feature_importance_values[col_index] += all_lags_importance[i]
            
            self.feature_importances_ = feature_importance_values
        except Exception as e:
            print(f"Warning calculating FI: {e}")

    # ... (Giữ nguyên phần predict và predict_history cũ của bạn) ...
    # Bạn chỉ cần copy lại phần predict/predict_history vào đây
    
    def predict(self, X, use_valid_history=True):
        # ... (Code cũ của bạn)
        # Lưu ý: cần đảm bảo logic predict cũng dùng .astype(float) nếu cần
        from tqdm import tqdm
        
        # ... [Logic check history] ...
        if self.last_y_train is None: raise RuntimeError("Call fit() first")

        y_hist_vals = self.last_y_train.values.astype(float).flatten()
        X_hist_vals = self.last_X_train.values.astype(float)

        if use_valid_history and self.last_y_valid is not None:
            y_hist_vals = np.concatenate([y_hist_vals, self.last_y_valid.values.astype(float).flatten()])
            X_hist_vals = np.vstack([X_hist_vals, self.last_X_valid.values.astype(float)])

        X_future_vals = X.values.astype(float) if hasattr(X, "values") else np.array(X, dtype=float)
        
        # ... [Phần còn lại của hàm predict giống hệt code cũ] ...
        # (Để ngắn gọn tôi không paste lại toàn bộ logic predict trừ khi bạn cần)
        
        # Placeholder cho phần predict cũ để class hoàn chỉnh về syntax
        # Logic Loop Predict Rolling ở đây
        n_predict = len(X_future_vals)
        preds = []
        current_y_hist = y_hist_vals.copy()
        input_len = self.input_chunk_length
        
        # Loop dự báo nhanh (bỏ tqdm cho gọn demo, bạn có thể thêm lại)
        for i in range(n_predict):
            idx_start_X = len(X_hist_vals) + i - input_len
            idx_end_X = len(X_hist_vals) + i
            
            # Cần xử lý nối mảng X_hist và X_future nếu idx vượt quá len history
            # Logic đơn giản hóa:
            if i == 0:
                 X_full = np.vstack([X_hist_vals, X_future_vals])
            
            x_window = X_full[idx_start_X : idx_end_X].flatten()
            y_window = current_y_hist[-input_len:]
            
            features = np.concatenate([x_window, y_window]).reshape(1, -1)
            val = self.model.predict(features)[0]
            preds.append(val)
            current_y_hist = np.append(current_y_hist, val)
            
        return np.array(preds)

    def predict_history(self, type="valid", verbose=True):
        # ... [Copy y nguyên code predict_history cũ của bạn] ...
        # Chỉ cần thêm .astype(float) vào chỗ lấy .values
        import numpy as np
        from tqdm import tqdm
        
        if type == "train":
            X_full = self.last_X_train.values.astype(float)
            y_full = self.last_y_train.values.astype(float).flatten()
            start_idx = self.input_chunk_length
            end_idx = len(X_full) - self.output_chunk_length + 1
        else: # valid
            X_train = self.last_X_train.values.astype(float)
            y_train = self.last_y_train.values.astype(float).flatten()
            X_valid = self.last_X_valid.values.astype(float)
            y_valid = self.last_y_valid.values.astype(float).flatten()
            
            X_full = np.vstack([X_train, X_valid])
            y_full = np.concatenate([y_train, y_valid])
            start_idx = len(X_train)
            end_idx = len(X_full) - self.output_chunk_length + 1
            
        preds = []
        input_len = self.input_chunk_length
        iterator = range(start_idx, end_idx)
        if verbose: iterator = tqdm(iterator, desc=f"Predict History {type}")
        
        for curr_idx in iterator:
            x_window = X_full[curr_idx - input_len : curr_idx].flatten()
            y_window = y_full[curr_idx - input_len : curr_idx]
            features = np.concatenate([x_window, y_window]).reshape(1, -1)
            
            p = self.model.predict(features)
            val = p[0][0] if len(p.shape) > 1 else p[0]
            preds.append(val)
            
        return np.array(preds)

class XGBWrapper:
    """
    Wrapper sklearn-like cho XGBRegressor.
    Fix lỗi: Setting an array element with a sequence.
    """

    def __init__(self, model=None, param_dict=None, input_chunk_length=7, output_chunk_length=1):
        if model is not None:
            self.model = model
        elif param_dict is not None:
            self.param_dict = param_dict
            self.model = XGBRegressor(**self.param_dict)
        else:
            raise ValueError("Phải truyền model hoặc param_dict")

        self.input_chunk_length  = input_chunk_length
        self.output_chunk_length = output_chunk_length
        
        # Lưu trữ data
        self.last_y_train = None
        self.last_X_train = None
        self.last_y_valid = None
        self.last_X_valid = None 
        self.feature_cols = []

    # ------------------- sklearn API -------------------
    def get_params(self, deep=True):
        return {"input_chunk_length"  : self.input_chunk_length,
                "output_chunk_length" : self.output_chunk_length,
                **self.model.get_params()}

    def set_params(self, **params):
        if "input_chunk_length" in params:
            self.input_chunk_length = params.pop("input_chunk_length")
        if "output_chunk_length" in params:
            self.output_chunk_length = params.pop("output_chunk_length")
        
        self.param_dict.update(**params)
        self.model = XGBRegressor(**self.param_dict)
        
        # Lưu trữ data
        self.last_y_train = None
        self.last_X_train = None
        self.last_y_valid = None
        self.last_X_valid = None 
        self.feature_cols = []
        
        return self

    # ------------------- FIT -------------------
    def fit(self, X, y, X_valid=None, y_valid=None, verbose=True, fi=False):
        # 1. Chuẩn hóa về Pandas để dễ xử lý
        if isinstance(X, np.ndarray):
            X = pd.DataFrame(X, columns=[f'feat_{i}' for i in range(X.shape[1])])
        if isinstance(y, np.ndarray):
            y = pd.Series(y)
            
        self.feature_cols = X.columns.tolist()

        # 2. Lưu trữ dữ liệu gốc
        self.last_y_train = y
        self.last_X_train = X
        self.last_y_valid = y_valid
        self.last_X_valid = X_valid

        # 3. Chuẩn bị dữ liệu Numpy
        # Ép kiểu float để tránh lỗi object (như bài trước)
        X_vals = X.values.astype(np.float64)
        y_vals = y.values.astype(np.float64).flatten()
        
        # --- FIX LỖI INDEX ERROR TẠI ĐÂY ---
        # Kiểm tra độ lệch độ dài
        len_x = len(X_vals)
        len_y = len(y_vals)
        
        if len_x != len_y:
            print(f"⚠️ Cảnh báo: Kích thước X ({len_x}) và y ({len_y}) không khớp!")
            # Lấy độ dài chung nhỏ nhất để an toàn
            min_len = min(len_x, len_y)
            X_vals = X_vals[:min_len]
            y_vals = y_vals[:min_len]
            n_samples = min_len
        else:
            n_samples = len_x
        # -----------------------------------

        n_features = X_vals.shape[1]
        
        # Tính toán index loop
        start_idx = self.input_chunk_length
        end_idx = n_samples - self.output_chunk_length + 1
        num_train_samples = end_idx - start_idx
        
        if num_train_samples <= 0:
            raise ValueError(f"Dữ liệu quá ngắn ({n_samples}) so với input ({self.input_chunk_length}) + output ({self.output_chunk_length}).")

        # Pre-allocate arrays
        input_dim = (self.input_chunk_length * n_features) + self.input_chunk_length
        X_train_rf = np.zeros((num_train_samples, input_dim), dtype=np.float64)
        
        if self.output_chunk_length == 1:
            y_train_rf = np.zeros((num_train_samples,), dtype=np.float64)
        else:
            y_train_rf = np.zeros((num_train_samples, self.output_chunk_length), dtype=np.float64)

        if verbose: 
            print(f">> Tạo Rolling Data: Samples={num_train_samples} | Input={self.input_chunk_length} | Output={self.output_chunk_length}")

        # 4. Tạo dữ liệu Rolling
        idx = 0
        for i in range(start_idx, end_idx):
            # --- Input ---
            # X lags
            x_window = X_vals[i - self.input_chunk_length : i].flatten()
            # y lags
            y_window = y_vals[i - self.input_chunk_length : i]
            
            X_train_rf[idx] = np.concatenate([x_window, y_window])
            
            # --- Target ---
            target = y_vals[i : i + self.output_chunk_length]
            
            # (Safety check thừa nhưng an toàn)
            if len(target) == 0:
                raise IndexError(f"Lỗi logic: Target rỗng tại index {i}. Check len(y)={len(y_vals)}")

            if self.output_chunk_length == 1:
                y_train_rf[idx] = target[0] 
            else:
                y_train_rf[idx] = target
                
            idx += 1

        # 5. Fit
        if verbose: print(f">> Fitting RandomForest...")
        self.model.fit(X_train_rf, y_train_rf)
        
        if fi:
            self._calculate_feature_importance(n_features)
                
        return self

    def _calculate_feature_importance(self, n_features_X):
        """Tách logic FI ra hàm riêng cho gọn"""
        try:
            all_lags_importance = self.model.feature_importances_
            input_len = self.input_chunk_length
            
            # Chỉ tính phần X lags (bỏ qua phần y lags ở đuôi vector input)
            # Vector input cấu trúc: [X_lag_1...X_lag_N, y_lag_1...y_lag_N]
            # Độ dài phần X là: n_features_X * input_len
            n_lags_X = n_features_X * input_len
            
            feature_importance_values = np.zeros(n_features_X, dtype=float)
            
            for i in range(n_lags_X):
                col_index = i % n_features_X
                feature_importance_values[col_index] += all_lags_importance[i]
            
            self.feature_importances_ = feature_importance_values
        except Exception as e:
            print(f"Warning calculating FI: {e}")

    # ... (Giữ nguyên phần predict và predict_history cũ của bạn) ...
    # Bạn chỉ cần copy lại phần predict/predict_history vào đây
    
    def predict(self, X, use_valid_history=True):
        # ... (Code cũ của bạn)
        # Lưu ý: cần đảm bảo logic predict cũng dùng .astype(float) nếu cần
        from tqdm import tqdm
        
        # ... [Logic check history] ...
        if self.last_y_train is None: raise RuntimeError("Call fit() first")

        y_hist_vals = self.last_y_train.values.astype(float).flatten()
        X_hist_vals = self.last_X_train.values.astype(float)

        if use_valid_history and self.last_y_valid is not None:
            y_hist_vals = np.concatenate([y_hist_vals, self.last_y_valid.values.astype(float).flatten()])
            X_hist_vals = np.vstack([X_hist_vals, self.last_X_valid.values.astype(float)])

        X_future_vals = X.values.astype(float) if hasattr(X, "values") else np.array(X, dtype=float)
        
        # ... [Phần còn lại của hàm predict giống hệt code cũ] ...
        # (Để ngắn gọn tôi không paste lại toàn bộ logic predict trừ khi bạn cần)
        
        # Placeholder cho phần predict cũ để class hoàn chỉnh về syntax
        # Logic Loop Predict Rolling ở đây
        n_predict = len(X_future_vals)
        preds = []
        current_y_hist = y_hist_vals.copy()
        input_len = self.input_chunk_length
        
        # Loop dự báo nhanh (bỏ tqdm cho gọn demo, bạn có thể thêm lại)
        for i in range(n_predict):
            idx_start_X = len(X_hist_vals) + i - input_len
            idx_end_X = len(X_hist_vals) + i
            
            # Cần xử lý nối mảng X_hist và X_future nếu idx vượt quá len history
            # Logic đơn giản hóa:
            if i == 0:
                 X_full = np.vstack([X_hist_vals, X_future_vals])
            
            x_window = X_full[idx_start_X : idx_end_X].flatten()
            y_window = current_y_hist[-input_len:]
            
            features = np.concatenate([x_window, y_window]).reshape(1, -1)
            val = self.model.predict(features)[0]
            preds.append(val)
            current_y_hist = np.append(current_y_hist, val)
            
        return np.array(preds)

    def predict_history(self, type="valid", verbose=True):
        # ... [Copy y nguyên code predict_history cũ của bạn] ...
        # Chỉ cần thêm .astype(float) vào chỗ lấy .values
        import numpy as np
        from tqdm import tqdm
        
        if type == "train":
            X_full = self.last_X_train.values.astype(float)
            y_full = self.last_y_train.values.astype(float).flatten()
            start_idx = self.input_chunk_length
            end_idx = len(X_full) - self.output_chunk_length + 1
        else: # valid
            X_train = self.last_X_train.values.astype(float)
            y_train = self.last_y_train.values.astype(float).flatten()
            X_valid = self.last_X_valid.values.astype(float)
            y_valid = self.last_y_valid.values.astype(float).flatten()
            
            X_full = np.vstack([X_train, X_valid])
            y_full = np.concatenate([y_train, y_valid])
            start_idx = len(X_train)
            end_idx = len(X_full) - self.output_chunk_length + 1
            
        preds = []
        input_len = self.input_chunk_length
        iterator = range(start_idx, end_idx)
        if verbose: iterator = tqdm(iterator, desc=f"Predict History {type}")
        
        for curr_idx in iterator:
            x_window = X_full[curr_idx - input_len : curr_idx].flatten()
            y_window = y_full[curr_idx - input_len : curr_idx]
            features = np.concatenate([x_window, y_window]).reshape(1, -1)
            
            p = self.model.predict(features)
            val = p[0][0] if len(p.shape) > 1 else p[0]
            preds.append(val)
            
        return np.array(preds)

# 4. LOADING MODELS

In [ ]:
!pip show torch

In [ ]:
reg_model = dict()
scaler_x = dict()
scaler_y = dict()

for name in station.keys():
    if model_type == "ML":
        reg_model[name] = joblib.load(model_dir + f"{model_aliases}_trained_{name}.pkl")
    else:
        # # Load lại wrapper
        reg_model[name] = joblib.load(model_dir + f"{model_aliases}_trained_wrapper_{name}.pkl")
        
        # # Load lại model
        if current_model == "NBEATS":
            reg_model[name].model = NBEATSModel.load(model_dir + f"{model_aliases}_trained_{name}.pt",weights_only=False)
        elif current_model == "TFT":
            reg_model[name].model = TFTModel.load(model_dir + f"{model_aliases}_trained_{name}.pt",weights_only=False)

    
    scaler_x[name] = joblib.load(model_dir.replace("trained_models","checkpoints") + f"scaler_x_{name}.pkl")
    scaler_y[name] = joblib.load(model_dir.replace("trained_models","checkpoints") + f"scaler_y_{name}.pkl")
    

In [ ]:
station_fit   = dict()
station_valid = dict()
station_pred  = dict()

for name in station.keys():
    print(f"🔸 Trạm: {name}")
    station_fit[name]    = pd.DataFrame(data    = reg_model[name].predict_history(type="train"), 
                                        index   = y_train_encoded[name]["TEMP_max"].iloc[reg_model[name].get_params()["input_chunk_length"]:].index, 
                                        columns = [y_train_encoded[name]["TEMP_max"].name])
    if model_type == "ML":
        station_valid[name]  = pd.DataFrame(data    = reg_model[name].predict_history(type="valid"), 
                                            index   = y_valid_encoded[name]["TEMP_max"].index, 
                                            columns = [y_valid_encoded[name]["TEMP_max"].name])
    else:
        station_valid[name]  = pd.DataFrame(data    = reg_model[name].predict_history(type="valid"), 
                                            index   = y_valid_encoded[name]["TEMP_max"].iloc[reg_model[name].get_params()["input_chunk_length"]:].index, 
                                            columns = [y_valid_encoded[name]["TEMP_max"].name])
        
    station_pred[name]   = pd.DataFrame(data    = reg_model[name].predict(X_test_encoded[name][feature_col]), 
                                        index   = y_test_encoded[name]["TEMP_max"].index, 
                                        columns = [y_test_encoded[name]["TEMP_max"].name])

# 5. METRICS

In [ ]:
y_true_valid = dict()
for name in station.keys():
    if model_type == "ML":
        y_true_valid[name] = scaler_y[name].inverse_transform(y_valid_encoded[name][["TEMP_max"]])
    else:
        y_true_valid[name] = scaler_y[name].inverse_transform(y_valid_encoded[name][["TEMP_max"]].iloc[reg_model[name].get_params()["input_chunk_length"]:])

## 5.1. r2_score

In [ ]:
r2 = dict({"train" : dict(),
           "valid" : dict(),
           "test"  : dict()})

In [ ]:
for name in station.keys():
    print(f"🔸 Trạm: {name}")
    r2["train"][name] = r2_score(y_true      = scaler_y[name].inverse_transform(y_train_encoded[name][["TEMP_max"]].iloc[reg_model[name].get_params()["input_chunk_length"]:]),
                                 y_pred      = scaler_y[name].inverse_transform(station_fit[name]),
                                 multioutput = "uniform_average")
    print(r2["train"][name])

In [ ]:
for name in station.keys():
    print(f"🔸 Trạm: {name}")
    r2["valid"][name] = r2_score(y_true      = y_true_valid[name],
                                 y_pred      = scaler_y[name].inverse_transform(station_valid[name]),
                                 multioutput = "uniform_average")
    print(r2["valid"][name])

In [ ]:
for name in station.keys():
    print(f"🔸 Trạm: {name}")
    r2["test"][name] = r2_score(y_true      = scaler_y[name].inverse_transform(y_test_encoded[name][["TEMP_max"]]),
                                y_pred      = scaler_y[name].inverse_transform(station_pred[name]),
                                multioutput = "uniform_average")
    print(r2["test"][name])

## 5.2. mean_absolute_error

In [ ]:
mae = dict({"train" : dict(),
           "valid" : dict(),
           "test"  : dict()})

In [ ]:
for name in station.keys():
    print(f"🔸 Trạm: {name}")
    mae["train"][name] = mean_absolute_error(y_true      = scaler_y[name].inverse_transform(y_train_encoded[name][["TEMP_max"]].iloc[reg_model[name].get_params()["input_chunk_length"]:]),
                                             y_pred      = scaler_y[name].inverse_transform(station_fit[name]),
                                             multioutput = "uniform_average")
    print(mae["train"][name])

In [ ]:
for name in station.keys():
    print(f"🔸 Trạm: {name}")
    mae["valid"][name] = mean_absolute_error(y_true      = y_true_valid[name],
                                             y_pred      = scaler_y[name].inverse_transform(station_valid[name]),
                                             multioutput = "uniform_average")
    print(mae["valid"][name])

In [ ]:
for name in station.keys():
    print(f"🔸 Trạm: {name}")
    mae["test"][name] = mean_absolute_error(y_true      = scaler_y[name].inverse_transform(y_test_encoded[name][["TEMP_max"]]),
                                            y_pred      = scaler_y[name].inverse_transform(station_pred[name]),
                                            multioutput = "uniform_average")
    print(mae["test"][name])

## 5.3. mean_squared_error

In [ ]:
mse = dict({"train" : dict(),
            "valid" : dict(),
            "test"  : dict()})

In [ ]:
for name in station.keys():
    print(f"🔸 Trạm: {name}")
    mse["train"][name] = mean_squared_error(y_true      = scaler_y[name].inverse_transform(y_train_encoded[name][["TEMP_max"]].iloc[reg_model[name].get_params()["input_chunk_length"]:]),
                                            y_pred      = scaler_y[name].inverse_transform(station_fit[name]),
                                            multioutput = "uniform_average")
    print(mse["train"][name])

In [ ]:
for name in station.keys():
    print(f"🔸 Trạm: {name}")
    mse["valid"][name] = mean_squared_error(y_true      = y_true_valid[name],
                                            y_pred      = scaler_y[name].inverse_transform(station_valid[name]),
                                            multioutput = "uniform_average")
    print(mse["valid"][name])

In [ ]:
for name in station.keys():
    print(f"🔸 Trạm: {name}")
    mse["test"][name] = mean_squared_error(y_true      = scaler_y[name].inverse_transform(y_test_encoded[name][["TEMP_max"]]),
                                           y_pred      = scaler_y[name].inverse_transform(station_pred[name]),
                                           multioutput = "uniform_average")
    print(mse["test"][name])

## 5.4. root_mean_squared_error

In [ ]:
rmse = dict({"train" : dict(),
             "valid" : dict(),
             "test"  : dict()})

In [ ]:
for name in station.keys():
    print(f"🔸 Trạm: {name}")
    rmse["train"][name] = root_mean_squared_error(y_true      = scaler_y[name].inverse_transform(y_train_encoded[name][["TEMP_max"]].iloc[reg_model[name].get_params()["input_chunk_length"]:]),
                                                 y_pred      = scaler_y[name].inverse_transform(station_fit[name]),
                                                 multioutput = "uniform_average")
    print(rmse["train"][name])

In [ ]:
for name in station.keys():
    print(f"🔸 Trạm: {name}")
    rmse["valid"][name] = root_mean_squared_error(y_true      = y_true_valid[name],
                                                 y_pred      = scaler_y[name].inverse_transform(station_valid[name]),
                                                 multioutput = "uniform_average")
    print(rmse["valid"][name])

In [ ]:
for name in station.keys():
    print(f"🔸 Trạm: {name}")
    rmse["test"][name] = root_mean_squared_error(y_true      = scaler_y[name].inverse_transform(y_test_encoded[name][["TEMP_max"]]),
                                                y_pred      = scaler_y[name].inverse_transform(station_pred[name]),
                                                multioutput = "uniform_average")
    print(rmse["test"][name])

## 5.5. mean_absolute_percentage_error

In [ ]:
mape = dict({"train" : dict(),
             "valid" : dict(),
             "test"  : dict()})

In [ ]:
for name in station.keys():
    print(f"🔸 Trạm: {name}")
    mape["train"][name] = mean_absolute_percentage_error(y_true      = scaler_y[name].inverse_transform(y_train_encoded[name][["TEMP_max"]].iloc[reg_model[name].get_params()["input_chunk_length"]:]),
                                                         y_pred      = scaler_y[name].inverse_transform(station_fit[name]),
                                                         multioutput = "uniform_average")
    print(mape["train"][name])

In [ ]:
for name in station.keys():
    print(f"🔸 Trạm: {name}")
    mape["valid"][name] = mean_absolute_percentage_error(y_true      = y_true_valid[name],
                                                         y_pred      = scaler_y[name].inverse_transform(station_valid[name]),
                                                         multioutput = "uniform_average")
    print(mape["valid"][name])

In [ ]:
for name in station.keys():
    print(f"🔸 Trạm: {name}")
    mape["test"][name] = mean_absolute_percentage_error(y_true      = scaler_y[name].inverse_transform(y_test_encoded[name][["TEMP_max"]]),
                                                        y_pred      = scaler_y[name].inverse_transform(station_pred[name]),
                                                        multioutput = "uniform_average")
    print(mape["test"][name])

## 5.6. metric_dataframe

In [ ]:
# Tổ chức metrics cho mô hình hiện tại
metrics  = dict({
                "MAE" : mae,
                "MSE" : mse,
                "RMSE": rmse,
                "MAPE": mape,
                "R2"  : r2,
                })

# ========== LƯU METRICS VÀO DICT TỔNG ==========
# Khởi tạo dict lưu tất cả mô hình (nếu chưa có)
if 'all_models_metrics' not in globals():
    all_models_metrics = dict({})

# Lưu metrics của mô hình hiện tại
all_models_metrics[current_model] = metrics.copy()

# ========== TẠO DATAFRAME ==========
station_order  = list(['NB', 'TH', 'DH', 'QN', 'TSN', 'CaMau'])
station_sorted = dict({key: station[key] for key in station_order if key in station})

print(f"✅ Đã lưu metrics cho mô hình: {current_model}")
print(f"📊 Tổng số mô hình đã lưu: {len(all_models_metrics)}")
print(f"   Danh sách: {all_models_metrics.keys()}")

In [ ]:
def create_metric_dataframe(metrics, station_order):
    data = list()
    for set in next(iter(metrics.values())).keys():
        for station in station_order:
            row = dict({"set": set, "station": station})

            for name, values in metrics.items():
                if station in values[set]:
                    value = values[set][station]

                    if name.lower().startswith("r2"):
                        value = round(value * 100, 4)
                    else:
                        value = round(value, 4)
                    row[name] = value

            data.append(row)

    df = pd.DataFrame(data)
    return df

# Hiển thị DataFrame cho mô hình hiện tại
print(f"\n📋 Bảng metrics cho mô hình {current_model}:")
create_metric_dataframe(metrics, station_order)

# 6. Visualizing

In [ ]:
from cycler import cycler
import matplotlib as mpl

mpl.rcParams['axes.prop_cycle'] = cycler(color=['blue'])

## 6.1. Actual vs Predict

In [ ]:
def actual_vs_predict_line_plot(station, target, y_train_encoded, station_fit, scaler_y,
                                set_name = None):
    import matplotlib.pyplot as plt
    import seaborn as sns
    import tensorflow as tf

    for name in station.keys():
        plt.figure(figsize=(15, 5))
        # true_fit
        if set_name == "kiểm tra" or (model_type == "ML" and set_name == "xác thực"):
            sns.lineplot(x     = station_fit[name].index,
                         y     = scaler_y[name].inverse_transform(y_train_encoded[name][[target]]).ravel(),
                         color = 'blue',
                         label = 'Actual',
                         #  ax    = axes[i],
                         linewidth=1.5)
        else:
            sns.lineplot(x     = station_fit[name].index,
                         y     = scaler_y[name].inverse_transform(y_train_encoded[name][[target]].iloc[reg_model[name].get_params()["input_chunk_length"]:]).ravel(),
                         color = 'blue',
                         label = 'Actual',
                         #  ax    = axes[i],
                         linewidth=1.5)
        # fit
        sns.lineplot(x     = station_fit[name].index,
                     y     = tf.squeeze(scaler_y[name].inverse_transform(station_fit[name])),
                     color = 'orange',
                     label = 'Predict',
                     #  ax    = axes[i],
                     linewidth=1.5,
                     alpha=0.8)
        plt.title(f"Tập {set_name}: Actual vs Predict - {name}", fontsize=14, fontweight='bold')
        plt.grid()
        plt.tight_layout()
        plt.show()

actual_vs_predict_line_plot(station         = station,
                            target          = "TEMP_max",
                            y_train_encoded = y_train_encoded,
                            station_fit     = station_fit,
                            scaler_y        = scaler_y,
                            set_name        = "huấn luyện")

actual_vs_predict_line_plot(station         = station,
                            target          = "TEMP_max",
                            y_train_encoded = y_valid_encoded,
                            station_fit     = station_valid,
                            scaler_y        = scaler_y,
                            set_name        = "xác thực")

actual_vs_predict_line_plot(station         = station,
                            target          = "TEMP_max",
                            y_train_encoded = y_test_encoded,
                            station_fit     = station_pred,
                            scaler_y        = scaler_y,
                            set_name        = "kiểm tra")

In [ ]:
def actual_vs_predict_scatter_plot(station, src_name, target, y_train_encoded, station_fit, scaler_y,
                                   set_name = None):
    import matplotlib.pyplot as plt
    import seaborn as sns
    import tensorflow as tf
    import numpy as np

    num_cols = len(station_fit.keys())

    # Xác định số hàng và số cột hợp lý
    ncols = 2  # Số biểu đồ trên mỗi hàng
    nrows = int(np.ceil(num_cols / ncols))  # Tính số hàng cần thiết

    fig, ax = plt.subplots(ncols              = ncols, 
                           nrows              = nrows, 
                           figsize            = (5*ncols, 4*nrows),
                           constrained_layout = True)  # auto căn chỉnh
    ax = ax.flatten()  # chuyển mảng 2 chiều thành 1 chiều để dễ duyệt
    count = list(["a", "b", "c", "d", "e", "f"])
    for i, name in enumerate(station.keys(), 0):
        # fit
        if set_name == "kiểm tra" or (model_type == "ML" and set_name == "xác thực"):
            sns.scatterplot(x  = scaler_y[name].inverse_transform(y_train_encoded[name][[target]]).ravel(),
                            y  = tf.squeeze(scaler_y[name].inverse_transform(station_fit[name])),
                            ax = ax[i])
            ax[i].set_xlabel("Thực tế")
            ax[i].set_ylabel("Dự đoán")
            # true_fit
            ax[i].plot(scaler_y[name].inverse_transform(y_train_encoded[name][[target]]),
                       scaler_y[name].inverse_transform(y_train_encoded[name][[target]]),
                       "r--")
        else:
            sns.scatterplot(x  = scaler_y[name].inverse_transform(y_train_encoded[name][[target]].iloc[reg_model[name].get_params()["input_chunk_length"]:]).ravel(),
                            y  = tf.squeeze(scaler_y[name].inverse_transform(station_fit[name])),
                            ax = ax[i])
            ax[i].set_xlabel("Thực tế")
            ax[i].set_ylabel("Dự đoán")
            # true_fit
            ax[i].plot(scaler_y[name].inverse_transform(y_train_encoded[name][[target]].iloc[reg_model[name].get_params()["input_chunk_length"]:]),
                       scaler_y[name].inverse_transform(y_train_encoded[name][[target]].iloc[reg_model[name].get_params()["input_chunk_length"]:]),
                       "r--")
        ax[i].set_title(f"({count[i]}) {src_name[name]}")
    plt.suptitle(f"""So sánh giá trị thực tế và giá trị dự đoán nhiệt độ cực đại\ntại các trạm trên tập {set_name}""", 
                 fontsize   = 15, 
                 fontweight = 'bold', 
                 ha         = 'center')
    # plt.savefig(f'Model: {current_model} - Set: {set_name} - Actual vs Predict.png', dpi=300, bbox_inches='tight')
    plt.show()

actual_vs_predict_scatter_plot(station         = station,
                               src_name        = src_name,
                               target          = "TEMP_max",
                               y_train_encoded = y_train_encoded,
                               station_fit     = station_fit,
                               scaler_y        = scaler_y,
                               set_name        = "huấn luyện")

actual_vs_predict_scatter_plot(station         = station,
                               src_name        = src_name,
                               target          = "TEMP_max",
                               y_train_encoded = y_valid_encoded,
                               station_fit     = station_valid,
                               scaler_y        = scaler_y,
                               set_name        = "xác thực")

actual_vs_predict_scatter_plot(station         = station,
                               src_name        = src_name,
                               target          = "TEMP_max",
                               y_train_encoded = y_test_encoded,
                               station_fit     = station_pred,
                               scaler_y        = scaler_y,
                               set_name        = "kiểm tra")

## 6.2. Metrics

In [ ]:
def plot_metric_panel(stations, station_names, metric,
                      color   = None,
                      label   = None,
                      display = False,
                      ax      = None):
        import numpy as np
        
        x = np.arange(len(stations))
        if display is True:
            ax.plot(x, metric, marker='o', color=color, label=label, linewidth=1.8)
            ax.set_xticks(x)
            ax.set_xticklabels([station_names.get(s, s) for s in stations])
            ax.set_xlabel('Trạm khí tượng')            

            if label == 'R2':
                ax.set_ylabel('R2 (%)')
                ax.set_title(f'(a) Hệ số xác định R2')    
            else:
                ax.set_ylabel('Chỉ số lỗi')
                ax.set_title(f'(b) Chỉ số lỗi')
            ax.legend()
            ax.grid(axis='both', linestyle='--')            

def plot_metrics_in_all_station(station, station_names, metrics_dict,
                                datasets     = dict({
                                                    # 'train': 'huấn luyện',
                                                    # 'valid': 'xác thực',
                                                    # 'test' : 'kiểm tra'
                                                    }),                        
                                metric_names = list([
                                                    #   'R2', 
                                                    #   'MAE', 
                                                    #   'MSE', 
                                                    #   'RMSE', 
                                                    #   'MAPE'
                                                    ]),
                                display  = True):
    import numpy as np
    import matplotlib.pyplot as plt

    for ds, info in datasets.items():
        fig, ax = plt.subplots(2, 2, figsize=(15, 10))
        stations = list(station.keys())
        if "R2" in metric_names:
            plot_metric_panel(stations      = stations,
                              station_names = station_names,
                              metric        = [float(metrics_dict.get("R2", {}).get(ds, {}).get(s, np.nan) * 100.0) for s in stations],
                              color         = '#1f77b4',
                              label         = 'R2',
                              display       = display,
                              ax            = ax[0][0])

        if "MAE" in metric_names:
            plot_metric_panel(stations      = stations,
                              station_names = station_names,
                              metric        = [metrics_dict.get("MAE", {}).get(ds, {}).get(s, np.nan) for s in stations],
                              color         = "#d62728",
                              label         = 'MAE',
                              display       = display,
                              ax            = ax[0][1])
        if "MSE" in metric_names:
            plot_metric_panel(stations      = stations,
                              station_names = station_names,
                              metric        = [metrics_dict.get("MSE", {}).get(ds, {}).get(s, np.nan) for s in stations],
                              color         = '#1f77b4',
                              label         = 'MSE',
                              display       = display,
                              ax            = ax[0][1])

        if "RMSE" in metric_names:
            plot_metric_panel(stations      = stations,
                              station_names = station_names,
                              metric        = [metrics_dict.get("RMSE", {}).get(ds, {}).get(s, np.nan) for s in stations],
                              color         = '#ff7f0e',
                              label         = 'RMSE',
                              display       = display,
                              ax            = ax[1][0])

        if "MAPE" in metric_names:
            plot_metric_panel(stations      = stations,
                              station_names = station_names,
                              metric        = [metrics_dict.get("MAPE", {}).get(ds, {}).get(s, np.nan) for s in stations],
                              color         = '#2ca02c',
                              label         = 'MAPE',
                              display       = display,
                              ax            = ax[1][1])

            fig.suptitle(f'Các chỉ số đánh giá mô hình trên tập {info}', 
                         fontsize   = 15, 
                         fontweight = 'bold',
                         ha         = 'center')
            plt.tight_layout()
            # plt.savefig(f'Model: {current_model} - Set: {info} - Metrics.png', dpi=300, bbox_inches='tight')
            plt.show()
            
plot_metrics_in_all_station(station       = station_sorted, 
                            station_names = src_name,
                            metrics_dict  = metrics,
                            datasets      = dict({
                                                  'train': 'huấn luyện',
                                                  'valid': 'xác thực',
                                                  'test' : 'kiểm tra'
                                                  }),
                            metric_names  = list([
                                                  'R2', 
                                                  'MAE', 
                                                  'MSE', 
                                                  'RMSE', 
                                                  'MAPE'
                                                  ]),
                            display       = True)

## 6.3. Feature Important

In [ ]:
num_cols = len(station_fit.keys())

# Xác định số hàng và số cột hợp lý
ncols = 2  # Số biểu đồ trên mỗi hàng
nrows = int(np.ceil(num_cols / ncols))  # Tính số hàng cần thiết

fig, ax = plt.subplots(ncols              = ncols, 
                       nrows              = nrows, 
                       figsize            = (5*ncols, 4*nrows),
                       constrained_layout = True)  # auto căn chỉnh
ax = ax.flatten()  # chuyển mảng 2 chiều thành 1 chiều để dễ duyệt

count = list(["a", "b", "c", "d", "e", "f"])
for i, name in enumerate(station.keys(), 0):
    # Concat train và test data cho trạm này
    train_data = pd.concat([X_train_encoded[name], X_test_encoded[name]], axis=0)
    
    # Tạo dictionary feature importance
    fi = dict({k: v for k,v in sorted(zip(train_data[feature_col].columns,
                                          reg_model[name].feature_importances_),
                                      key     = lambda x : x[1],
                                      reverse = True)})
    
    # Plot
    sns.barplot(fi, orient='h', ax=ax[i])
    ax[i].grid(True, axis='x')
    ax[i].set_title(f"({count[i]}) {src_name[name]}")

plt.suptitle("Mức độ quan trọng của đặc trưng tại các trạm khí tượng", 
             fontsize   = 15, 
             fontweight = 'bold', 
             ha         = 'center')
plt.tight_layout()
# plt.savefig(f'Model: {current_model} - Feature Importance.png', dpi=300, bbox_inches='tight')
plt.show()

## 6.4. SHAP

In [ ]:
# def shap_summary_plot(station, src_name, shap_vals, features_to_plot, feature_col,
#                       set_name,
#                       col_width ,
#                       row_height,
#                       ncols     ):
#     import numpy as np
#     import matplotlib.pyplot as plt
#     import shap

#     # Tính layout
#     num_cols   = len(station.keys())
#     nrows      = int(np.ceil(num_cols / ncols))

#     fig, ax = plt.subplots(nrows   = nrows, 
#                            ncols   = ncols, 
#                            figsize = (col_width * ncols, row_height * nrows), 
#                            squeeze = False)
#     ax = ax.flatten() # chuyển mảng 2 chiều thành 1 chiều để dễ duyệt

#     for i, name in enumerate(station.keys()):
#         print(f"shap_val.{name}:", shap_vals[name].shape)
#         print("features.shape:", features_to_plot[name][feature_col].shape)
#         plt.sca(ax[i])
#         count = list(["a", "b", "c", "d", "e", "f"])
#         shap.summary_plot(shap_vals[name],
#                           features      = features_to_plot[name][feature_col],
#                           feature_names = features_to_plot[name][feature_col].columns,
#                           show          = False,
#                           plot_size     = (col_width, row_height))

#         ax[i].set_title(f"({count[i]}) {src_name[name]}")
#         ax[i].tick_params(axis='both', which='both')

#     plt.suptitle(f"SHAP Summary tại các trạm khí tượng trên tập {set_name}",
#                  fontsize   = 15, 
#                  fontweight = 'bold', 
#                  ha         = 'center')
#     plt.tight_layout()
#     plt.show()

## 6.5. Comparasion Models

In [ ]:
def comparasion_model(station, station_names, metrics_dict, 
                      datasets     = None,                        
                      metric_names = None,
                      ncols        = 2):
    import numpy as np
    import matplotlib.pyplot as plt

    x = np.arange(len(station))
    x_labels = [station_names.get(s, s) for s in station]
    
    marker_list = ['o', 's', '^', 'D', 'v', 'P', '*', 'X', '<', '>']
    colors = ['red','blue','green','purple','orange','brown','pink','gray','olive','cyan']

    for ds, info in datasets.items():
        n_metrics = len(metric_names)
        nrows = int(np.ceil(n_metrics / ncols))
        fig, axes = plt.subplots(ncols=ncols, nrows=nrows,
                                 figsize=(7*ncols, 5*nrows),
                                 constrained_layout=True
                                 )
        axes = axes.flatten() if n_metrics > 1 else axes

        for idx, metric in enumerate(metric_names):
            for i, model in enumerate(metrics_dict.keys()):
                vals = list([metrics_dict[model].get(metric, {}).get(ds, {}).get(s, np.nan) * (100 if metric == 'R2' else 1)
                             for s in station])
                    
                axes[idx].plot(x, vals, marker=marker_list[i], label=model, color=colors[i % len(colors)])
                axes[idx].set_title(f'{metric}', fontsize=12, fontweight='bold')
                axes[idx].set_xticks(x)
                axes[idx].set_xticklabels(x_labels)
                axes[idx].legend()
                axes[idx].grid(axis='both', linestyle='--')

        for j in range(n_metrics, len(axes)):
            axes[j].axis('off')

        fig.suptitle(f'So sánh độ chính xác dự báo của các mô hình trên tập {info}', fontsize=16, fontweight='bold', y=1.03)
        # plt.savefig(f'Set: {info} - comparasion_model.png', dpi=300, bbox_inches='tight')
        plt.show()

comparasion_model(station       = station_order,
                  station_names = src_name,
                  metrics_dict  = all_models_metrics,
                  datasets = dict({
                                    'train': 'huấn luyện',
                                    'valid': 'xác thực',
                                    'test' : 'kiểm tra'
                                  }),                        
                  metric_names  = list([
                                        'R2', 
                                        'MAE', 
                                        'MSE', 
                                        'RMSE', 
                                        'MAPE'
                                        ]),
                  ncols         = 2)